# Day 1.8 — Pivotal Exercise: Complete the Manual Agent Loop

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.

## Why this mechanism matters

A tool-using agent is an application-controlled loop. The model proposes either a tool request or a final answer; Python validates, dispatches, records the observation, and decides whether another step is allowed. Day 1.5 built this loop with you; here you write it alone.

## Contract

Implement `run_agent`. Reject unknown tools with `ValueError`, append every tool result to `messages` as `{"role": "tool", "name": ..., "content": ...}`, return the final text, and raise `RuntimeError` when `max_steps` is exhausted.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def run_agent(model, tools, messages, max_steps=4):
    """Run a bounded observe-dispatch-append loop and return final text.

    model(messages) returns either
        {"type": "final", "text": "..."}                       -> return the text
        {"type": "tool", "name": "...", "arguments": {...}}    -> run the tool, append, loop
    """
    # TODO: repeat for at most max_steps
    # TODO: ask model(messages) for the next response
    # TODO: return response["text"] when type == "final"
    # TODO: validate the tool name, run tools[name](**arguments)
    # TODO: append {"role": "tool", "name": ..., "content": ...} to messages
    # TODO: after the loop, raise RuntimeError("step limit reached")
    raise NotImplementedError("Complete the agent loop")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    calls = []
    def add(a, b):
        calls.append((a, b))
        return a + b

    # Case 1: one tool request, then a final answer.
    responses = iter([
        {"type": "tool", "name": "add", "arguments": {"a": 2, "b": 3}},
        {"type": "final", "text": "The result is 5."},
    ])
    history = [{"role": "user", "content": "Add 2 and 3"}]
    assert run_agent(lambda messages: next(responses), {"add": add}, history) == "The result is 5."
    assert calls == [(2, 3)], "the tool must run exactly once with the model's arguments"
    assert any(item.get("role") == "tool" for item in history), "the tool result must be appended"

    # Case 2: an unknown tool must be rejected BEFORE anything runs.
    bad = iter([{"type": "tool", "name": "delete_everything", "arguments": {}}])
    try:
        run_agent(lambda messages: next(bad), {"add": add}, [{"role": "user", "content": "x"}])
    except ValueError:
        pass
    else:
        raise AssertionError("an unknown tool name must raise ValueError")

    # Case 3: a model that never answers must be stopped by the step limit.
    forever = lambda messages: {"type": "tool", "name": "add", "arguments": {"a": 1, "b": 1}}
    try:
        run_agent(forever, {"add": add}, [{"role": "user", "content": "x"}], max_steps=3)
    except RuntimeError:
        pass
    else:
        raise AssertionError("the loop must raise RuntimeError after max_steps")
    print("PASS: dispatch, unknown-tool rejection, and step limit all behave correctly")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def run_agent(model, tools, messages, max_steps=4):
    """Run a bounded observe-dispatch-append loop and return final text."""
    for step in range(1, max_steps + 1):                 # bounded: the loop can never run forever
        response = model(messages)                        # 1. ask the model what to do next
        if response["type"] == "final":                   # 2a. it answered -> we are done
            return response["text"]
        if response["type"] != "tool":                    # anything else is a protocol error
            raise ValueError(f"Unexpected response type: {response['type']!r}")
        name = response["name"]
        if name not in tools:                             # 2b. validate BEFORE executing anything
            raise ValueError(f"Unknown tool requested: {name!r}")
        result = tools[name](**response["arguments"])     # 3. the HOST runs the function, not the model
        messages.append({"role": "tool", "name": name, "content": str(result)})  # 4. record the observation
        print(f"step {step}: ran {name}{response['arguments']} -> {result}")
    raise RuntimeError(f"Stopped after {max_steps} steps without a final answer")  # 5. safe termination

print("Reference run_agent defined. Re-run the check cell above to see PASS.")

## Explain

**Why must the application, rather than the model, own tool execution and termination?**

<details><summary>Show answer</summary>

The model only emits text that <em>looks like</em> a request. Only the host can check the tool exists, validate the arguments, decide whether it is allowed, actually run it, and count steps. If the model owned termination, a confused or malicious prompt could loop forever or call anything.

</details>

**Why is the unknown-tool check placed before <code>tools[name](...)</code> and not after?**

<details><summary>Show answer</summary>

Once a function has run, its side effects have happened. Validation must come first so a bad request is rejected with zero effects.

</details>